# Does cropping to the food help classification? — dataset `04_denoise_clahe_dino`

**Question.** Dataset 03 removed the background with a Grounded-SAM mask. Dataset 04 only crops each image to Grounding DINO's food boxes and keeps every real pixel (no mask).
Does a box crop help, and is it better or worse than the mask? The same notebook runs on every dataset; only `DATASET` changes:

| dataset | preprocessing |
|---|---|
| `01_original` | resized originals (baseline) |
| `02_denoise_clahe` | 01 + wavelet denoise (BayesShrink) + CLAHE on the L channel of LAB |
| `03_denoise_clahe_gsam` | 02 + Grounded-SAM mask: food kept, background grey (124,116,104), cropped to the food |
| `04_denoise_clahe_dino` | 02 + crop to the union of Grounding DINO's food boxes; real pixels kept, no mask |

The comparisons that answer the question are **02 → 04** (does the crop help?) and **03 vs 04** (mask or box?). Section 11 runs them once the other datasets' results are in the same `results/` folder.

**Data.** 10 classes × (578 train / 193 val / 192 test) = 9,630 images, the same images in the same split as 01–03 (same manifest).
Wrong-label, non-food and duplicate images were already removed (`meta/removed.csv`). **Never re-split**: the folders are the split.

**Models** (same architecture, same tuning grid and same training budget as the 03 run, see 0.1):

| # | model | what is tuned (on val only) | train/val graph |
|---|---|---|---|
| 1 | CLIP ViT-B/32 zero-shot (baseline) | prompt template | val score per template (nothing is trained) |
| 2 | Classical: handcrafted features → Logistic Regression / SVM (RBF) | `C`, `gamma` | validation curve over `C` |
| 3 | CNN from scratch | depth (number of conv blocks), epochs (early stopping) | learning curves + train/val vs depth |
| 4 | ResNet-50 transfer (ImageNet) | how much to unfreeze (`fc` / `layer4` / `all`), epochs | learning curves + train/val vs unfreeze level |

**Metrics.** top-1 / top-3 / top-5 accuracy, macro precision / recall / F1, per-class AP and mAP, 10×10 confusion matrix,
correct and wrong test examples, Grad-CAM. Model selection uses **val macro-F1**. Every config tried is scored on val and test and saved as soon as it is trained
(`results/<dataset>/all_models/`); the test score of the configs that were not chosen is for the report only.

### How to run
- **VS Code (local, Apple M1 → MPS):** unzip so that `data/dataset/04_denoise_clahe_dino/` and `data/dataset/meta/` exist, pick the `.venv` kernel, *Run All*.
- **Colab (GPU):** *Runtime → Change runtime type → T4 GPU*. Upload `04_denoise_clahe_dino.zip` to `MyDrive/food_classify/`. *Run All* (it mounts Drive and unzips).
  Keep the 03 results in `MyDrive/food_classify/results/03_denoise_clahe_gsam/` (they are already there if 03 ran on Colab): section 10.2 and section 11 compare 03 and 04 from them.
- First run `RUN_MODE = "smoke"` (20 images per class, 2 epochs) to check that every cell works, then `RUN_MODE = "full"`.
- Everything trained is saved; re-running a cell loads it instead of training again (`RETRAIN = True` to force).

Rough full-run time, from the 03 run on a Colab T4: about 2 h (SVM grid ≈ 45 min because Colab has 2 CPU cores, CNN ≈ 45 min, ResNet-50 ≈ 25 min, the rest ≈ 10 min). M1 ≈ 3–4 h.

## 0. Configuration

### 0.1 Fair-comparison rules used here
1. **Same architecture and same search space** for a model on 01 / 02 / 03 / 04 (the grids below). Each dataset may pick its own best value from that grid,
   chosen by val macro-F1 with the same early-stopping rule. Same budget, same rule, so differences come from the data, not from extra tuning.
2. **Same input pipeline everywhere:** pad to square with the grey fill → resize to 224×224 → ImageNet normalisation (CLIP uses its own mean/std).
   Padding instead of center-crop because many GSAM and DINO crops are wider than 4:3 and a center crop would cut food off.
3. **No colour jitter** in augmentation, because colour and contrast are what dataset 02 changes. Augmentation is only a mild random crop + horizontal flip, on train only.
4. Same seed, same split. Test is never used to choose anything.
5. The settings in the next cell are the ones the 03 run used. Change a value for every dataset or for none; changing it for 04 alone breaks the 03 vs 04 comparison.

In [ ]:
# ==== the only cell you normally edit ====
DATASET = "04_denoise_clahe_dino"   # "01_original" | "02_denoise_clahe" | "03_denoise_clahe_gsam" | "04_denoise_clahe_dino"
RUN_MODE = "smoke"                  # "smoke": 20 imgs/class/split, 2 epochs, to test the pipeline | "full": the real run
SEED = 42
RETRAIN = False                     # True: ignore saved checkpoints / grid results and train again

# ---- from here on: identical for every dataset (same as the 03 run) ----
IMG_SIZE = 224
PAD_FILL = (124, 116, 104)          # same grey as the GSAM background (used to pad every dataset to a square)
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
LABEL_SMOOTHING = 0.1

CLASSICAL_FEATURES = "handcrafted"  # "handcrafted" (HSV hist + HOG + LBP) | "resnet50" (frozen ImageNet embeddings)
LOGREG_C = [1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100]
SVM_C = [0.1, 1, 10, 100, 1000]
SVM_GAMMA_MULT = [0.3, 1, 3]        # x 1/n_features (= sklearn "scale" after standardising)

CNN_GRID = [dict(n_blocks=n, base=32, dropout=0.3) for n in (3, 4, 5)]
CNN_TRAIN = dict(max_epochs=40, patience=6, lr=1e-3, weight_decay=5e-4)

RESNET_GRID = [dict(unfreeze="fc", lr=1e-3), dict(unfreeze="layer4", lr=3e-4), dict(unfreeze="all", lr=1e-4)]
RESNET_TRAIN = dict(max_epochs=15, patience=4, weight_decay=1e-4)

CLIP_ID = "openai/clip-vit-base-patch32"
TEMPLATE_SETS = {
    "classname": ["{}"],
    "a photo of": ["a photo of {}."],
    "food photo": ["a photo of {}, a type of food."],
    "ensemble": ["a photo of {}, a type of food.", "a close-up photo of {}.", "a photo of a plate of {}.",
                 "a good photo of {}.", "a cropped photo of {}."],
}

EXPECTED_MANIFEST_SHA256 = "79eb1d1cf8a475b8deaa2a301acc64a1c3a334fd4a289c8d45af91c359c3c2a4"
SMOKE_PER_CLASS = 20
if RUN_MODE == "smoke":
    CNN_TRAIN = {**CNN_TRAIN, "max_epochs": 2}
    RESNET_TRAIN = {**RESNET_TRAIN, "max_epochs": 2}
    LOGREG_C, SVM_C, SVM_GAMMA_MULT = [1e-2, 1], [1, 10], [1]

In [ ]:
# ==== paths (local VS Code or Colab) ====
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/food_classify")   # upload <DATASET>.zip here
    DATA_ROOT = Path("/content/data/dataset")
    if not (DATA_ROOT / DATASET).exists():
        import zipfile
        with zipfile.ZipFile(DRIVE_DIR / f"{DATASET}.zip") as z:
            z.extractall(DATA_ROOT)
    OUT_ROOT = DRIVE_DIR / "results"                               # on Drive, so it survives a disconnect
else:
    here = Path.cwd().resolve()
    PROJECT = next((p for p in [here, *here.parents] if (p / "data" / "dataset" / DATASET).exists()), None)
    assert PROJECT is not None, f"data/dataset/{DATASET} not found above {here}: unzip the dataset into data/dataset/"
    DATA_ROOT = PROJECT / "data" / "dataset"
    OUT_ROOT = PROJECT / "results"
    os.environ.setdefault("TORCH_HOME", str(PROJECT / "checkpoints" / "torch"))  # reuse downloaded weights
    os.environ.setdefault("HF_HOME", str(PROJECT / "checkpoints" / "hf"))

META = DATA_ROOT / "meta"
OUT = (OUT_ROOT / DATASET) if RUN_MODE == "full" else (OUT_ROOT / "smoke" / DATASET)
CKPT_DIR, FIG_DIR = OUT / "checkpoints", OUT / "figures"
for d in (OUT, CKPT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("data   :", DATA_ROOT / DATASET)
print("results:", OUT)

In [ ]:
import json, time, random, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support, f1_score, average_precision_score, confusion_matrix
from sklearn.preprocessing import label_binarize

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"                    # fp16 helps on CUDA; it did not help on the M1
NON_BLOCKING = DEVICE.type == "cuda"
NUM_WORKERS = 0 if sys.platform == "darwin" else 2  # macOS Jupyter can't pickle classes defined in a notebook into workers
pd.set_option("display.precision", 4)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def savefig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=120, bbox_inches="tight")


def finish(fig, title, name):
    """Lay out a grid figure with its title above the first row (no overlap), save, show."""
    fig.tight_layout(rect=(0, 0, 1, 1 - 0.55 / fig.get_figheight()))
    fig.suptitle(title, y=1 - 0.12 / fig.get_figheight())
    savefig(fig, name)
    plt.show()


seed_everything()
print(f"python {sys.version.split()[0]} ({sys.executable}) | torch {torch.__version__} | torchvision {torchvision.__version__} | device {DEVICE}")

## 1. Data checks
Before training: is it the same data as the rest of the team, are the classes balanced, is there leakage between splits, and what did the DINO crop do?

In [ ]:
sha = hashlib.sha256((META / "manifest.csv").read_bytes()).hexdigest()
assert sha == EXPECTED_MANIFEST_SHA256, f"manifest differs from the team's ({sha}): wrong zip?"

df_all = pd.read_csv(META / "manifest.csv")
df_all["path"] = [str(DATA_ROOT / DATASET / r) for r in df_all.relpath]
missing = [p for p in df_all.path if not os.path.exists(p)]
assert not missing, f"{len(missing)} images in the manifest are missing, e.g. {missing[:3]}"
on_disk = {str(p.relative_to(DATA_ROOT / DATASET)) for p in (DATA_ROOT / DATASET).glob("*/*/*.jpg")}
assert on_disk == set(df_all.relpath), "folder contents and manifest differ"
assert (df_all.relpath.str.split("/").str[0] == df_all.split).all() and \
       (df_all.relpath.str.split("/").str[1] == df_all.label).all(), "folder and manifest split/label disagree"

CLASSES = sorted(df_all.label.unique())
K = len(CLASSES)
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}
df_all["y"] = df_all.label.map(CLS2IDX)

key = df_all.label + "/" + df_all.relpath.str.split("/").str[-1]
assert not key.duplicated().any(), "the same image id appears in more than one split"

counts = df_all.pivot_table(index="label", columns="split", values="relpath", aggfunc="count")[["train", "val", "test"]]
counts.loc["total"] = counts.sum()
display(counts)
assert counts.drop("total").nunique().eq(1).all(), "classes are not balanced"
print("split ratio:", (counts.loc["total"] / counts.loc["total"].sum()).round(3).to_dict())

df = df_all.groupby(["split", "label"], group_keys=False).head(SMOKE_PER_CLASS) if RUN_MODE == "smoke" else df_all
splits = {s: df[df.split == s].reset_index(drop=True) for s in ("train", "val", "test")}
y_split = {s: v.y.values for s, v in splits.items()}
print({s: len(v) for s, v in splits.items()}, "images used in this run")

In [ ]:
removed = pd.read_csv(META / "removed.csv")
removed["label"] = removed.key.str.split("/").str[0]
print(f"{len(removed)} images were removed during cleaning:")
display(pd.crosstab(removed.label, removed.reason, margins=True))

In [ ]:
# near-duplicate check across splits (8x8 average hash). Report only: the split is fixed by the team.
def ahash(path):
    g = np.asarray(Image.open(path).convert("L").resize((8, 8), Image.BILINEAR), dtype=np.float32)
    return np.packbits(g > g.mean()).tobytes()

df_all["ahash"] = [ahash(p) for p in df_all.path]
dup = df_all[df_all.duplicated("ahash", keep=False)].sort_values("ahash")
cross = dup.groupby("ahash").filter(lambda g: g.split.nunique() > 1)
print(f"{dup.ahash.nunique()} hash groups with >1 image; {cross.ahash.nunique()} of them span different splits")
if len(cross):
    show = cross.groupby("ahash").head(2).head(12)
    fig, axes = plt.subplots(1, len(show), figsize=(2 * len(show), 2.4), squeeze=False)
    for ax, (_, r) in zip(axes[0], show.iterrows()):
        ax.imshow(Image.open(r.path)); ax.set_title(f"{r.split}\n{r.relpath.split('/')[-1]}", fontsize=7); ax.axis("off")
    plt.show()

In [ ]:
# 6 random training images per class
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(K, 6, figsize=(12, 2 * K))
for i, c in enumerate(CLASSES):
    rows = df_all[(df_all.split == "train") & (df_all.label == c)]
    for j, p in enumerate(rng.choice(rows.path.values, 6, replace=False)):
        ax = axes[i, j]; ax.imshow(Image.open(p)); ax.set_xticks([]); ax.set_yticks([])
        if j == 0: ax.set_ylabel(c, rotation=0, ha="right", fontsize=9)
finish(fig, f"{DATASET}: train samples", "samples")

sizes = np.array([Image.open(p).size for p in df_all.path])
aspect = sizes.max(1) / sizes.min(1)
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].hist(sizes[:, 0], bins=40, alpha=0.6, label="width"); axes[0].hist(sizes[:, 1], bins=40, alpha=0.6, label="height")
axes[0].set_title("image size (px)"); axes[0].legend()
axes[1].hist(aspect, bins=40); axes[1].set_title(f"aspect ratio (long/short); {np.mean(aspect > 4 / 3):.0%} wider than 4:3")
plt.tight_layout(); plt.show()

In [ ]:
# what the crop did. box_area = union of the food boxes as a fraction of the 02 image; fallback = no box or < 5%, so the 02 image is used
DINO_STATS = META / "dino_crop_stats.csv"
HAS_DINO = DATASET.startswith("04") and DINO_STATS.exists()
if HAS_DINO:
    dn = pd.read_csv(DINO_STATS)
    dn["label"] = dn.relpath.str.split("/").str[1]
    dn["barely_cropped"] = dn.box_area >= 0.9
    display(dn.groupby("split").agg(n=("fallback", "size"), fallback_rate=("fallback", "mean"), box_area_median=("box_area", "median"),
                                    keeps_90pct=("barely_cropped", "mean"), boxes_median=("n_boxes", "median")).loc[["train", "val", "test"]])
    fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
    axes[0].hist(dn[dn.fallback == 0].box_area, bins=50); axes[0].set_title("box area (fraction of the 02 image), cropped images")
    axes[1].hist(dn.n_boxes.clip(upper=8), bins=np.arange(-0.5, 9.5)); axes[1].set_title("food boxes per image (8 = 8 or more)")
    per_cls = dn.groupby("label").agg(box_area=("box_area", "median"), fallbacks=("fallback", "sum"))
    per_cls.box_area.plot.bar(ax=axes[2]); axes[2].set_title("median box area per class")
    for i, n in enumerate(per_cls.fallbacks): axes[2].text(i, 0.01, f"fb={n}", ha="center", rotation=90, color="white")
    plt.tight_layout(); savefig(fig, "dino_stats"); plt.show()
    print(f"{dn.barely_cropped.mean():.0%} of the images keep >= 90% of the 02 image, so the crop changes them little. "
          f"{int(dn.fallback.sum())} fallbacks: {dn[dn.fallback == 1].label.value_counts().to_dict()}")

    show = pd.concat([dn[dn.fallback == 0].nsmallest(6, "box_area"),
                      dn[dn.fallback == 1].sample(min(6, int(dn.fallback.sum())), random_state=SEED)])
    fig, axes = plt.subplots(1, len(show), figsize=(2 * len(show), 2.8), squeeze=False)
    for ax, (_, r) in zip(axes[0], show.iterrows()):
        tag = "fallback" if r.fallback else f"box {r.box_area:.2f}"
        ax.imshow(Image.open(DATA_ROOT / DATASET / r.relpath)); ax.set_title(f"{tag}\n{r.label}", fontsize=8); ax.axis("off")
    finish(fig, "tightest crops (left) and fallbacks (right)", "dino_tightest_and_fallbacks")
else:
    print("no DINO crop stats for this dataset; section 10 will be skipped")

## 2. Input pipeline
Pad to square (grey) → 224×224. Train adds a mild random crop (70–100% of the area) and a horizontal flip.
The same transform is used for CNN and ResNet-50; CLIP and the classical features use the same pad + resize with their own normalisation / size.

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
CLIP_MEAN, CLIP_STD = [0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711]


def pad_to_square(img, fill=PAD_FILL):
    w, h = img.size
    s = max(w, h)
    out = Image.new("RGB", (s, s), fill)
    out.paste(img, ((s - w) // 2, (s - h) // 2))
    return out


train_tf = T.Compose([T.Lambda(pad_to_square),
                      T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0), ratio=(0.9, 1.1)),
                      T.RandomHorizontalFlip(),
                      T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
eval_tf = T.Compose([T.Lambda(pad_to_square), T.Resize((IMG_SIZE, IMG_SIZE)),
                     T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
clip_tf = T.Compose([T.Lambda(pad_to_square), T.Resize((IMG_SIZE, IMG_SIZE)),
                     T.ToTensor(), T.Normalize(CLIP_MEAN, CLIP_STD)])


def show_tensor(t, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    """Normalised CHW tensor -> HWC image in [0, 1] for plotting."""
    x = t.cpu() * torch.tensor(std)[:, None, None] + torch.tensor(mean)[:, None, None]
    return x.clamp(0, 1).permute(1, 2, 0).numpy()


class FoodDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths, self.labels, self.transform = frame.path.tolist(), frame.y.tolist(), transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return self.transform(Image.open(self.paths[i]).convert("RGB")), self.labels[i]


def make_loader(split, transform, shuffle=False, batch_size=None):
    return DataLoader(FoodDataset(splits[split], transform), batch_size=batch_size or (BATCH_SIZE if shuffle else EVAL_BATCH_SIZE),
                      shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
                      persistent_workers=NUM_WORKERS > 0)


# what the models see
demo = splits["train"].sample(3, random_state=SEED)
fig, axes = plt.subplots(3, 5, figsize=(12, 7.5))
for row, (_, r) in zip(axes, demo.iterrows()):
    img = Image.open(r.path).convert("RGB")
    row[0].imshow(img); row[0].set_title(f"stored {img.size}", fontsize=8)
    row[1].imshow(show_tensor(eval_tf(img))); row[1].set_title("eval 224x224", fontsize=8)
    for ax in row[2:]:
        ax.imshow(show_tensor(train_tf(img))); ax.set_title("train aug", fontsize=8)
    for ax in row: ax.axis("off")
plt.tight_layout(); savefig(fig, "transforms"); plt.show()

## 3. Evaluation toolkit (shared by all models)
- **top-k accuracy**: true class among the k highest scores. With 10 classes top-5 is close to 100% for any decent model, so top-1 and macro-F1 are the headline numbers.
- **macro precision / recall / F1**: per-class values averaged with equal weight (the classes are balanced, so macro ≈ weighted).
- **AP** per class: area under the one-vs-rest precision–recall curve from the scores; **mAP** = mean over the 10 classes. It uses the ranking, not only the top-1 decision.
- **Confusion matrix**: yes, 10×10, as counts and row-normalised (the diagonal is per-class recall), plus a list of the most confused pairs.
- Saved per model in `results/<dataset>/<model>/`: `metrics.json`, `per_class_test.csv`, `predictions_{val,test}.csv` (one row per image with all 10 scores, needed for the cross-dataset test in section 11),
  and the **final model**: `model.pt` (CNN, ResNet-50, CLIP) or `model.joblib` (Logistic Regression, SVM, scaler included), plus `config.json`, which records what it is and how to rebuild it. Section 12 reloads any of them.
- **Every config tried** (each prompt set, each `C` / `gamma`, each depth, each unfreeze level) is scored on val and test as soon as it is trained and saved to `results/<dataset>/all_models/<config>/` (model + `config.json` + `metrics.json` + `scores.npz`). A re-run reads these instead of training or predicting again. The chosen model of each type is reported from those saved scores, and section 8.2 lists every config. The choice is still made on val only.

In [ ]:
RESULTS = {}   # model name -> {"val": {...}, "test": {...}, "config": {...}}


def topk_accuracy(y, scores, k):
    top = np.argsort(-scores, axis=1)[:, :k]
    return float((top == y[:, None]).any(axis=1).mean())


def compute_metrics(y, scores):
    y = np.asarray(y)
    pred = scores.argmax(1)
    p, r, f1, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    Y = label_binarize(y, classes=np.arange(K))
    ap = np.array([average_precision_score(Y[:, c], scores[:, c]) for c in range(K)])
    pc_p, pc_r, pc_f, support = precision_recall_fscore_support(y, pred, labels=np.arange(K), zero_division=0)
    per_class = pd.DataFrame({"precision": pc_p, "recall": pc_r, "f1": pc_f, "AP": ap, "support": support}, index=CLASSES)
    summary = {"top1": topk_accuracy(y, scores, 1), "top3": topk_accuracy(y, scores, 3), "top5": topk_accuracy(y, scores, 5),
               "macro_precision": float(p), "macro_recall": float(r), "macro_f1": float(f1), "mAP": float(np.nanmean(ap)), "n": int(len(y))}
    return summary, per_class


def macro_f1(y, scores):
    return f1_score(y, scores.argmax(1), average="macro", zero_division=0)


def plot_confusion(y, pred, title, name):
    cm = confusion_matrix(y, pred, labels=np.arange(K))
    cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
    fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))
    for ax, m, fmt, sub in [(axes[0], cm, "d", "counts"), (axes[1], cmn, ".2f", "row-normalised (diagonal = recall)")]:
        im = ax.imshow(m, cmap="Blues")
        ax.set_xticks(range(K), CLASSES, rotation=45, ha="right"); ax.set_yticks(range(K), CLASSES); ax.grid(False)
        for i in range(K):
            for j in range(K):
                ax.text(j, i, format(m[i, j], fmt), ha="center", va="center", fontsize=8,
                        color="white" if m[i, j] > m.max() / 2 else "black")
        ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"{title}: {sub}")
        fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout(); savefig(fig, f"{name}_confusion"); plt.show()
    off = [(CLASSES[i], CLASSES[j], int(cm[i, j])) for i in range(K) for j in range(K) if i != j and cm[i, j] > 0]
    return pd.DataFrame(sorted(off, key=lambda t: -t[2])[:8], columns=["true", "predicted", "count"])


def show_predictions(frame, y, scores, correct, title, name, n=12, is_prob=True):
    pred = scores.argmax(1)
    idx = np.flatnonzero((pred == y) if correct else (pred != y))
    if len(idx) == 0:
        print("no", "correct" if correct else "wrong", "predictions"); return
    idx = np.random.default_rng(SEED).choice(idx, min(n, len(idx)), replace=False)
    cols = 6
    rows = int(np.ceil(len(idx) / cols))
    row_h = 3.3 if correct else 3.7          # room for the 2- or 3-line titles above each image
    fig, axes = plt.subplots(rows, cols, figsize=(2.6 * cols, row_h * rows), squeeze=False)
    tag = "p" if is_prob else "s"
    for ax, i in zip(axes.ravel(), idx):
        top = np.argsort(-scores[i])[:2]
        ax.imshow(Image.open(frame.path[i]))
        txt = f"T: {CLASSES[y[i]]}\nP: {CLASSES[top[0]]} ({tag}={scores[i, top[0]]:.2f})"
        if not correct:
            txt += f"\n2nd: {CLASSES[top[1]]} ({scores[i, top[1]]:.2f})"
        ax.set_title(txt, fontsize=8, color="green" if correct else "red")
    for ax in axes.ravel(): ax.axis("off")
    finish(fig, title, name)


def save_predictions(name, split, y, scores):
    out = splits[split][["relpath", "label"]].copy()
    out["y_true"], out["y_pred"] = y, scores.argmax(1)
    out["pred_label"] = [CLASSES[i] for i in out.y_pred]
    for c in range(K):
        out[f"score_{CLASSES[c]}"] = scores[:, c]
    out.to_csv(OUT / name / f"predictions_{split}.csv", index=False)


def report_model(name, y_val, s_val, y_test, s_test, config, is_prob=True):
    """Final evaluation of a chosen model: val + test metrics, confusion matrix, examples, saved files."""
    (OUT / name).mkdir(exist_ok=True)
    sv, _ = compute_metrics(y_val, s_val)
    st, per_class = compute_metrics(y_test, s_test)
    RESULTS[name] = {"val": sv, "test": st, "config": config}
    display(pd.DataFrame([sv, st], index=["val", "test"]))
    display(per_class)
    display(plot_confusion(y_test, s_test.argmax(1), f"{name} (test)", name))
    show_predictions(splits["test"], y_test, s_test, True, f"{name}: correct test predictions", f"{name}_correct", is_prob=is_prob)
    show_predictions(splits["test"], y_test, s_test, False, f"{name}: wrong test predictions", f"{name}_wrong", is_prob=is_prob)
    save_predictions(name, "val", y_val, s_val)
    save_predictions(name, "test", y_test, s_test)
    per_class.to_csv(OUT / name / "per_class_test.csv")
    with open(OUT / name / "metrics.json", "w") as f:
        json.dump({"dataset": DATASET, "model": name, "seed": SEED, "run_mode": RUN_MODE, **RESULTS[name]}, f, indent=2, default=str)


def save_final(name, kind, model, arch=None, metrics=None):
    """Save a model as results/<dataset>/<name>/model.{pt,joblib} + config.json: the chosen model (after report_model) or any tried config.
    kind: "cnn" | "resnet50" | "clip" (torch, state_dict + architecture) or "sklearn" (whole fitted pipeline)."""
    import joblib
    d = OUT / name
    d.mkdir(parents=True, exist_ok=True)
    arch = json.loads(json.dumps(arch or {}, default=str))   # plain Python types, so torch.load(weights_only=True) can read it
    metrics = metrics or RESULTS.get(name, {})
    if kind == "sklearn":
        file = "model.joblib"
        joblib.dump(model, d / file)
    elif kind == "clip":
        file = "model.pt"
        torch.save({"kind": kind, "arch": arch, "classes": CLASSES, "text_emb": model.text_emb.cpu()}, d / file)
    else:
        file = "model.pt"
        torch.save({"kind": kind, "arch": arch, "classes": CLASSES,
                    "state_dict": {k: v.cpu() for k, v in model.state_dict().items()}}, d / file)
    config = {"dataset": DATASET, "model": name, "kind": kind, "file": file, "arch": arch, "classes": CLASSES,
              "preprocessing": {"pad_to_square_fill": PAD_FILL, "img_size": IMG_SIZE,
                                "normalize": "clip" if kind == "clip" else "imagenet",
                                "classical_features": CLASSICAL_FEATURES if kind == "sklearn" else None},
              "seed": SEED, "run_mode": RUN_MODE, "torch": torch.__version__,
              "val_macro_f1": metrics.get("val", {}).get("macro_f1"),
              "test_macro_f1": metrics.get("test", {}).get("macro_f1"),
              "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")}
    with open(d / "config.json", "w") as f:
        json.dump(config, f, indent=2, default=str)
    size = (d / file).stat().st_size / 1e6
    print(f"saved {d / file} ({size:.1f} MB) + config.json")


ALL_DIR = OUT / "all_models"   # every config tried: model + config.json + metrics.json + scores.npz
ALL_DIR.mkdir(exist_ok=True)
CONFIGS, CHOSEN = {}, {}       # config name -> its val/test metrics; model type -> the config chosen on val


def config_done(name, arch):
    """Metrics of a config already trained, scored and saved with this arch, else None (then it is trained / scored now)."""
    p = ALL_DIR / name / "metrics.json"
    if RETRAIN or not p.exists() or not (ALL_DIR / name / "scores.npz").exists():
        return None
    res = json.loads(p.read_text())
    if res.get("arch") != json.loads(json.dumps(arch, default=str)):
        return None
    CONFIGS[name] = res
    return res


def record_config(name, family, kind, model, arch, scores, train_f1):
    """Score one tried config on val and test, save it to all_models/<name>/ and remember it.
    scores: {"val": (y, s), "test": (y, s)}. Test is recorded for the report; choosing uses val only."""
    arch = json.loads(json.dumps(arch, default=str))
    res = {"model": name, "family": family, "kind": kind, "arch": arch, "train_f1": float(train_f1),
           **{sp: compute_metrics(*scores[sp])[0] for sp in ("val", "test")}}
    save_final(f"all_models/{name}", kind, model, arch, metrics=res)
    np.savez(ALL_DIR / name / "scores.npz", **{f"{sp}_{k}": v for sp, (y, s) in scores.items() for k, v in (("y", y), ("s", s))})
    with open(ALL_DIR / name / "metrics.json", "w") as f:
        json.dump({"dataset": DATASET, **res}, f, indent=2, default=str)
    CONFIGS[name] = res
    return res


def config_scores(name):
    """(y_val, s_val, y_test, s_test) of a recorded config, read back from all_models/<name>/scores.npz."""
    z = np.load(ALL_DIR / name / "scores.npz")
    return z["val_y"], z["val_s"], z["test_y"], z["test_s"]

In [ ]:
# shared PyTorch training loop (CNN and ResNet-50)
SCALER = torch.amp.GradScaler("cuda", enabled=USE_AMP)


def set_train_mode(model):
    """train() but keep frozen BatchNorm layers in eval mode, so a frozen backbone's running stats don't change."""
    model.train()
    for m in model.modules():
        if isinstance(m, nn.modules.batchnorm._BatchNorm) and not any(p.requires_grad for p in m.parameters()):
            m.eval()


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    set_train_mode(model) if training else model.eval()
    total, ys, ps = 0.0, [], []
    for x, y_cpu in loader:
        # non_blocking copies only on CUDA: on MPS with torch <= 2.12 they can deliver corrupted labels
        x, y = x.to(DEVICE, non_blocking=NON_BLOCKING), y_cpu.to(DEVICE, non_blocking=NON_BLOCKING)
        with torch.set_grad_enabled(training), torch.autocast(DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x)
            loss = F.cross_entropy(logits, y, label_smoothing=LABEL_SMOOTHING if training else 0.0)
        if training:
            optimizer.zero_grad(set_to_none=True)
            SCALER.scale(loss).backward()
            SCALER.step(optimizer)
            SCALER.update()
        total += loss.item() * len(y)
        ys.append(y_cpu); ps.append(logits.detach().float().softmax(1).cpu())
    y, p = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    return total / len(y), y, p


def fit(name, build, cfg, max_epochs, patience, lr, weight_decay):
    """Train with AdamW + cosine LR, keep the epoch with the best val macro-F1, stop after `patience` epochs without gain.
    Checkpoint files carry a hash of the config, so changing a grid value never reloads a stale model."""
    tag = hashlib.md5(str((cfg, max_epochs, patience, lr, weight_decay, LABEL_SMOOTHING, IMG_SIZE)).encode()).hexdigest()[:8]
    ckpt, hist_path = CKPT_DIR / f"{name}_{tag}.pt", CKPT_DIR / f"{name}_{tag}_history.csv"
    seed_everything()
    model = build().to(DEVICE)
    if ckpt.exists() and hist_path.exists() and not RETRAIN:
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        print(f"{name}: loaded saved model {ckpt.name}")
        return model, pd.read_csv(hist_path), ckpt
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{name}: {n_train / 1e6:.2f}M trainable params")
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)
    train_dl, val_dl = make_loader("train", train_tf, shuffle=True), make_loader("val", eval_tf)
    rows, best, best_epoch, wait = [], -1.0, 0, 0
    for epoch in range(1, max_epochs + 1):
        t0, lr_now = time.time(), opt.param_groups[0]["lr"]
        tr_loss, tr_y, tr_p = run_epoch(model, train_dl, opt)
        va_loss, va_y, va_p = run_epoch(model, val_dl)
        sched.step()
        row = dict(epoch=epoch, lr=lr_now, train_loss=tr_loss, val_loss=va_loss,
                   train_acc=(tr_p.argmax(1) == tr_y).mean(), val_acc=(va_p.argmax(1) == va_y).mean(),
                   train_f1=macro_f1(tr_y, tr_p), val_f1=macro_f1(va_y, va_p), sec=time.time() - t0)
        rows.append(row)
        print(f"  ep {epoch:2d} | loss {tr_loss:.3f}/{va_loss:.3f} | F1 {row['train_f1']:.3f}/{row['val_f1']:.3f} | {row['sec']:.0f}s")
        if row["val_f1"] > best:
            best, best_epoch, wait = row["val_f1"], epoch, 0
            torch.save(model.state_dict(), ckpt)
        else:
            wait += 1
            if wait >= patience:
                print(f"  early stop: no val gain for {patience} epochs")
                break
    hist = pd.DataFrame(rows)
    hist["best"] = hist.epoch == best_epoch
    hist.to_csv(hist_path, index=False)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return model, hist, ckpt


def load_trained(ckpt, build):
    model = build().to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return model.eval()


@torch.no_grad()
def predict(model, split, transform=None):
    _, y, p = run_epoch(model, make_loader(split, transform or eval_tf))
    return y, p


def plot_learning_curves(hists, title, name):
    """One row per config: loss and macro-F1 per epoch, train vs val, dashed line at the kept epoch."""
    fig, axes = plt.subplots(len(hists), 2, figsize=(12, 3 * len(hists)), squeeze=False)
    for row, (cfg_name, h) in zip(axes, hists.items()):
        best = h.loc[h.best, "epoch"].iloc[0]
        for ax, metric in zip(row, ("loss", "f1")):
            ax.plot(h.epoch, h[f"train_{metric}"], "o-", ms=3, label="train")
            ax.plot(h.epoch, h[f"val_{metric}"], "o-", ms=3, label="val")
            ax.axvline(best, ls="--", c="gray", lw=1)
            ax.set_title(f"{cfg_name}: {'loss' if metric == 'loss' else 'macro-F1'} (kept epoch {best})", fontsize=10)
            ax.set_xlabel("epoch"); ax.legend()
            ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    finish(fig, title, name)


def plot_grid_summary(table, x, title, name, logx=False):
    """Train vs val macro-F1 at the kept epoch for each config: the 'elbow' plot."""
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.plot(table[x].astype(str) if not logx else table[x], table.train_f1, "o-", label="train")
    ax.plot(table[x].astype(str) if not logx else table[x], table.val_f1, "o-", label="val")
    if logx: ax.set_xscale("log")
    best = table.val_f1.idxmax()
    ax.scatter([table[x].astype(str)[best] if not logx else table[x][best]], [table.val_f1[best]], s=160,
               facecolors="none", edgecolors="red", lw=2, label="chosen (best val)", zorder=5)
    ax.set_xlabel(x); ax.set_ylabel("macro-F1"); ax.set_title(title); ax.legend()
    plt.tight_layout(); savefig(fig, name); plt.show()

## 4. Model 1 — CLIP ViT-B/32 zero-shot (baseline)
No training: each class name goes into a text prompt, and the image is assigned to the class whose text embedding is closest.
A plain ImageNet ResNet can't do this (its output is the 1,000 ImageNet classes), which is why CLIP is the zero-shot baseline.
The only choice is the prompt template, picked on val. There is no train curve because nothing is fitted.

In [ ]:
from transformers import CLIPModel, AutoTokenizer

clip = CLIPModel.from_pretrained(CLIP_ID).to(DEVICE).eval()
for p in clip.parameters():
    p.requires_grad_(False)
clip_tok = AutoTokenizer.from_pretrained(CLIP_ID)


@torch.no_grad()
def clip_text_embeddings(templates):
    embs = []
    for c in CLASSES:
        t = clip_tok([tpl.format(c.replace("_", " ")) for tpl in templates], padding=True, return_tensors="pt").to(DEVICE)
        e = clip.text_projection(clip.text_model(input_ids=t["input_ids"], attention_mask=t["attention_mask"]).pooler_output)
        embs.append(F.normalize(F.normalize(e, dim=-1).mean(0), dim=0))
    return torch.stack(embs)


class CLIPZeroShot(nn.Module):
    """Image -> class logits (scaled cosine similarity to the class text embeddings)."""
    def __init__(self, clip, text_emb):
        super().__init__()
        self.clip = clip
        self.register_buffer("text_emb", text_emb)

    def forward(self, x):
        f = self.clip.visual_projection(self.clip.vision_model(pixel_values=x).pooler_output)
        return self.clip.logit_scale.exp() * F.normalize(f, dim=-1) @ self.text_emb.T


@torch.no_grad()
def clip_image_embeddings(split):
    cache = OUT / f"clip_img_{split}.pt"
    if cache.exists() and not RETRAIN:
        return torch.load(cache).to(DEVICE)
    feats = []
    for x, _ in make_loader(split, clip_tf):
        f = clip.visual_projection(clip.vision_model(pixel_values=x.to(DEVICE)).pooler_output)
        feats.append(F.normalize(f, dim=-1))
    feats = torch.cat(feats)
    torch.save(feats.cpu(), cache)
    return feats



img_emb = {s: clip_image_embeddings(s) for s in ("val", "test")}
scale = clip.logit_scale.exp()
rows = []
for tname, templates in TEMPLATE_SETS.items():
    name, arch = "clip_" + tname.replace(" ", "_"), {"clip_id": CLIP_ID, "templates": templates}
    res = config_done(name, arch)
    if res is None:
        model = CLIPZeroShot(clip, clip_text_embeddings(templates)).eval()
        sc = {sp: (y_split[sp], (scale * img_emb[sp] @ model.text_emb.T).softmax(1).cpu().numpy()) for sp in ("val", "test")}
        res = record_config(name, "clip_zeroshot", "clip", model, arch, sc, train_f1=float("nan"))   # nothing is trained
    rows.append({"templates": tname, "name": name, "val_top1": res["val"]["top1"], "val_f1": res["val"]["macro_f1"]})
clip_grid = pd.DataFrame(rows)
display(clip_grid)

fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.bar(clip_grid.templates, clip_grid.val_f1)
for i, v in enumerate(clip_grid.val_f1): ax.text(i, v, f"{v:.3f}", ha="center", va="bottom")
ax.set_ylabel("val macro-F1"); ax.set_title("CLIP zero-shot: prompt template (val)"); ax.set_ylim(0, 1)
plt.tight_layout(); savefig(fig, "clip_templates"); plt.show()

best_clip = clip_grid.loc[clip_grid.val_f1.idxmax()]
CHOSEN["clip_zeroshot"] = best_clip["name"]
clip_model = CLIPZeroShot(clip, clip_text_embeddings(TEMPLATE_SETS[best_clip.templates])).eval()   # kept for Grad-CAM
print("chosen templates:", best_clip.templates)
report_model("clip_zeroshot", *config_scores(best_clip["name"]), {"templates": TEMPLATE_SETS[best_clip.templates]})
save_final("clip_zeroshot", "clip", clip_model, {"clip_id": CLIP_ID, "templates": TEMPLATE_SETS[best_clip.templates]})

## 5. Model 2 — Classical: handcrafted features → Logistic Regression and SVM
An SVM or logistic regression can't learn well from 150,528 raw pixels, so each image becomes a fixed feature vector first:

| block | what it measures | dims |
|---|---|---|
| HSV colour histogram 8×8×8 | which colours are present | 512 |
| HOG (128×128 image, 16 px cells) | edge / shape directions | 1,764 |
| LBP uniform (8,1) + (16,2) | fine texture | 28 |

Background removal changes all three directly, so this is where the segmentation effect is most visible.
(`CLASSICAL_FEATURES = "resnet50"` swaps in frozen ImageNet ResNet-50 embeddings instead, a "linear probe". That is a useful extra, but it is no longer classical.)

**Validation curve / elbow.** For each `C` we fit on train and score both train and val. Small `C` = strong regularisation (underfits: both low).
Large `C` = weak regularisation (overfits: train → 1.0, val falls). Train usually stays above val instead of crossing it, so the point we keep is where **val peaks**,
just before the gap opens. That is the elbow.

Each config is fitted once and saved. Its val score is computed from the same scores that are reported (probabilities for LogReg, one-vs-rest decision values for the SVM), so the chosen config is the best one by the numbers in the report. The 03 notebook scored the SVM grid with `predict()` instead; on 03 both rules pick C = 10, gamma × 1.

In [ ]:
import cv2
from joblib import Parallel, delayed
from skimage.feature import hog, local_binary_pattern
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.exceptions import ConvergenceWarning
import joblib


def handcrafted(path):
    rgb = np.asarray(pad_to_square(Image.open(path).convert("RGB")).resize((128, 128), Image.BILINEAR))
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    color = cv2.calcHist([hsv], [0, 1, 2], None, [8, 8, 8], [0, 180, 0, 256, 0, 256]).ravel()
    color /= color.sum() + 1e-8
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    shape = hog(gray, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2), block_norm="L2-Hys")
    texture = []
    for p, r in ((8, 1), (16, 2)):
        h, _ = np.histogram(local_binary_pattern(gray, p, r, method="uniform"), bins=p + 2, range=(0, p + 2))
        texture.append(h / h.sum())
    return np.concatenate([color, shape, *texture]).astype(np.float32)


@torch.no_grad()
def resnet_embeddings(split):
    m = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
    m.fc = nn.Identity()
    m = m.to(DEVICE).eval()
    return np.concatenate([m(x.to(DEVICE)).cpu().numpy() for x, _ in make_loader(split, eval_tf)])


def load_features(split):
    cache = OUT / f"features_{CLASSICAL_FEATURES}_{split}.npz"
    if cache.exists() and not RETRAIN:
        z = np.load(cache, allow_pickle=True)
        if list(z["relpath"]) == splits[split].relpath.tolist():
            return z["X"]
    t0 = time.time()
    if CLASSICAL_FEATURES == "handcrafted":
        X = np.stack(Parallel(n_jobs=-1, batch_size=32)(delayed(handcrafted)(p) for p in splits[split].path))
    else:
        X = resnet_embeddings(split)
    np.savez(cache, X=X, relpath=splits[split].relpath.values)
    print(f"{split}: features {X.shape} in {time.time() - t0:.0f}s")
    return X


X = {s: load_features(s) for s in ("train", "val", "test")}
print({s: v.shape for s, v in X.items()})

In [ ]:
def sweep(family, make_model, grid, name_of):
    """Fit every config on train and score train / val / test; each fitted model is saved, so a re-run loads results instead of refitting."""
    rows = []
    for cfg in grid:
        name, arch = name_of(cfg), {"features": CLASSICAL_FEATURES, **cfg}
        res = config_done(name, arch)
        if res is None:
            t0 = time.time()
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", ConvergenceWarning)
                m = make_model(**cfg).fit(X["train"], y_split["train"])
            score = m.predict_proba if hasattr(m, "predict_proba") else m.decision_function
            res = record_config(name, family, "sklearn", m, arch, {sp: (y_split[sp], score(X[sp])) for sp in ("val", "test")},
                                train_f1=macro_f1(y_split["train"], score(X["train"])))
            print(f"  {cfg} -> train {res['train_f1']:.3f} | val {res['val']['macro_f1']:.3f} | {time.time() - t0:.0f}s")
        rows.append({**cfg, "name": name, "train_f1": res["train_f1"], "val_f1": res["val"]["macro_f1"]})
    return pd.DataFrame(rows)


def make_logreg(C):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000))


n_feat = X["train"].shape[1]
def make_svm(C, gamma_mult):
    return make_pipeline(StandardScaler(), SVC(C=C, gamma=gamma_mult / n_feat, kernel="rbf", decision_function_shape="ovr"))


print("Logistic Regression")
lr_grid = sweep("logreg", make_logreg, [dict(C=c) for c in LOGREG_C], lambda c: f"logreg_C{c['C']:g}")
display(lr_grid)
plot_grid_summary(lr_grid, "C", "Logistic Regression: validation curve", "logreg_curve", logx=True)

In [ ]:
print("SVM (RBF)")
svm_grid = sweep("svm", make_svm, [dict(C=c, gamma_mult=g) for g in SVM_GAMMA_MULT for c in SVM_C],
                 lambda c: f"svm_C{c['C']:g}_g{c['gamma_mult']:g}")
display(svm_grid.pivot(index="gamma_mult", columns="C", values="val_f1").style.background_gradient(cmap="Blues").format("{:.3f}"))

fig, ax = plt.subplots(figsize=(7, 4))
for i, (g, t) in enumerate(svm_grid.groupby("gamma_mult")):
    ax.plot(t.C, t.train_f1, "--", c=f"C{i}", label=f"train, gamma x{g}")
    ax.plot(t.C, t.val_f1, "o-", c=f"C{i}", label=f"val, gamma x{g}")
b = svm_grid.val_f1.idxmax()
ax.scatter([svm_grid.C[b]], [svm_grid.val_f1[b]], s=160, facecolors="none", edgecolors="red", lw=2, zorder=5, label="chosen")
ax.set_xscale("log"); ax.set_xlabel("C"); ax.set_ylabel("macro-F1"); ax.set_title("SVM (RBF): validation curves"); ax.legend(fontsize=7)
plt.tight_layout(); savefig(fig, "svm_curve"); plt.show()

In [ ]:
best_lr = lr_grid.loc[lr_grid.val_f1.idxmax()]
CHOSEN["logreg"] = best_lr["name"]
logreg = joblib.load(ALL_DIR / best_lr["name"] / "model.joblib")   # fitted in the sweep, not refitted
print("Logistic Regression, chosen C =", best_lr.C)
report_model("logreg", *config_scores(best_lr["name"]), {"features": CLASSICAL_FEATURES, "C": float(best_lr.C)})
save_final("logreg", "sklearn", logreg, {"features": CLASSICAL_FEATURES, "C": float(best_lr.C)})

In [ ]:
best_svm = svm_grid.loc[svm_grid.val_f1.idxmax()]
CHOSEN["svm"] = best_svm["name"]
svm = joblib.load(ALL_DIR / best_svm["name"] / "model.joblib")
print(f"SVM, chosen C = {best_svm.C}, gamma = {best_svm.gamma_mult} / n_features")
# SVC has no calibrated probabilities; the one-vs-rest decision values rank the classes, which is all top-k and AP need
report_model("svm", *config_scores(best_svm["name"]),
             {"features": CLASSICAL_FEATURES, "C": float(best_svm.C), "gamma_mult": float(best_svm.gamma_mult)}, is_prob=False)
save_final("svm", "sklearn", svm, {"features": CLASSICAL_FEATURES, "C": float(best_svm.C), "gamma_mult": float(best_svm.gamma_mult)})

## 6. Model 3 — CNN from scratch
VGG-style blocks of `[conv3×3-BN-ReLU] ×2 → maxpool`, channels doubling per block (32, 64, 128, …), then global average pooling → dropout → linear.
The tuning question is **depth** (3, 4 or 5 blocks); epochs are set by early stopping on val macro-F1.

Two kinds of graph:
1. **Learning curves** per config (loss and macro-F1 per epoch). Overfitting shows as train still improving while val flattens or gets worse. The dashed line marks the kept epoch.
2. **Train vs val against depth** at each config's kept epoch: the elbow plot.

Train scores are measured during the training pass (with augmentation and dropout on), which is standard and slightly underestimates them.

The table's `stopped` column says whether a config early-stopped or hit `max_epochs`. One that hit the limit was still improving, so its score is a lower bound (in the 03 run the 5-block CNN did). The limit stays at 40 so 04 is comparable with 03.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, n_blocks=4, base=32, dropout=0.3, n_classes=K):
        super().__init__()
        layers, c_in = [], 3
        for i in range(n_blocks):
            c = base * 2 ** i
            layers += [nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                       nn.Conv2d(c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                       nn.MaxPool2d(2)]
            c_in = c
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(dropout), nn.Linear(c_in, n_classes))

    def forward(self, x):
        return self.head(self.features(x))


cnn_hists, cnn_rows = {}, []
for cfg in CNN_GRID:
    name = f"cnn_d{cfg['n_blocks']}"
    model, hist, ckpt = fit(name, lambda: SimpleCNN(**cfg), cfg, **CNN_TRAIN)
    cnn_hists[name] = hist
    kept = hist[hist.best].iloc[0]
    if config_done(name, cfg) is None or ckpt.stat().st_mtime > (ALL_DIR / name / "metrics.json").stat().st_mtime:   # new or retrained
        record_config(name, "cnn", "cnn", model.eval(), cfg, {sp: predict(model, sp) for sp in ("val", "test")}, kept.train_f1)
    cnn_rows.append({**cfg, "name": name, "epoch": int(kept.epoch), "epochs_run": len(hist),
                     "stopped": "early stop" if len(hist) < CNN_TRAIN["max_epochs"] else "epoch limit",
                     "train_f1": kept.train_f1, "val_f1": kept.val_f1, "ckpt": str(ckpt)})
    del model
cnn_grid = pd.DataFrame(cnn_rows)
display(cnn_grid.drop(columns="ckpt"))
plot_learning_curves(cnn_hists, "CNN from scratch: learning curves", "cnn_learning_curves")
plot_grid_summary(cnn_grid, "n_blocks", "CNN: train vs val macro-F1 by depth", "cnn_depth")

In [ ]:
best_cnn = cnn_grid.loc[cnn_grid.val_f1.idxmax()]
CHOSEN["cnn"] = best_cnn["name"]
cnn_cfg = {"n_blocks": int(best_cnn.n_blocks), "base": int(best_cnn.base), "dropout": float(best_cnn.dropout)}
cnn_model = load_trained(best_cnn.ckpt, lambda: SimpleCNN(**cnn_cfg))   # kept for Grad-CAM; its scores were saved during the grid
print("chosen:", best_cnn["name"], cnn_cfg, "|", best_cnn.stopped, "at epoch", int(best_cnn.epoch))
report_model("cnn", *config_scores(best_cnn["name"]), {**cnn_cfg, **CNN_TRAIN, "epoch": int(best_cnn.epoch)})
save_final("cnn", "cnn", cnn_model, cnn_cfg)

## 7. Model 4 — ResNet-50 transfer learning (ImageNet weights)
New 10-class head (dropout 0.2 → linear). Tuning question: **how much of the pretrained network to fine-tune**.
- `fc`: only the head (feature extractor + classifier)
- `layer4`: last residual stage + head
- `all`: everything, with a smaller learning rate

Frozen BatchNorm layers stay in eval mode during training (`set_train_mode`); otherwise a "frozen" backbone would still drift.

In [ ]:
def make_resnet50(unfreeze, pretrained=True):
    m = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
    m.fc = nn.Sequential(nn.Dropout(0.2), nn.Linear(m.fc.in_features, K))
    for p in m.parameters():
        p.requires_grad_(False)
    for part in {"fc": [m.fc], "layer4": [m.layer4, m.fc], "all": [m]}[unfreeze]:
        for p in part.parameters():
            p.requires_grad_(True)
    return m


rn_hists, rn_rows = {}, []
for cfg in RESNET_GRID:
    name = f"resnet50_{cfg['unfreeze']}"
    model, hist, ckpt = fit(name, lambda: make_resnet50(cfg["unfreeze"]), cfg, lr=cfg["lr"], **RESNET_TRAIN)
    rn_hists[name] = hist
    kept = hist[hist.best].iloc[0]
    if config_done(name, cfg) is None or ckpt.stat().st_mtime > (ALL_DIR / name / "metrics.json").stat().st_mtime:   # new or retrained
        record_config(name, "resnet50", "resnet50", model.eval(), cfg, {sp: predict(model, sp) for sp in ("val", "test")}, kept.train_f1)
    rn_rows.append({**cfg, "name": name, "epoch": int(kept.epoch), "epochs_run": len(hist),
                    "stopped": "early stop" if len(hist) < RESNET_TRAIN["max_epochs"] else "epoch limit",
                    "train_f1": kept.train_f1, "val_f1": kept.val_f1, "ckpt": str(ckpt)})
    del model
rn_grid = pd.DataFrame(rn_rows)
display(rn_grid.drop(columns="ckpt"))
plot_learning_curves(rn_hists, "ResNet-50 transfer: learning curves", "resnet_learning_curves")
plot_grid_summary(rn_grid, "unfreeze", "ResNet-50: train vs val macro-F1 by fine-tuned depth", "resnet_unfreeze")

In [ ]:
best_rn = rn_grid.loc[rn_grid.val_f1.idxmax()]
CHOSEN["resnet50"] = best_rn["name"]
rn_model = load_trained(best_rn.ckpt, lambda: make_resnet50(best_rn["unfreeze"]))   # kept for Grad-CAM
print("chosen:", best_rn["name"], "|", best_rn.stopped, "at epoch", int(best_rn.epoch))
report_model("resnet50", *config_scores(best_rn["name"]),
             {"unfreeze": best_rn["unfreeze"], "lr": float(best_rn.lr), **RESNET_TRAIN, "epoch": int(best_rn.epoch)})
save_final("resnet50", "resnet50", rn_model, {"unfreeze": best_rn["unfreeze"]})

## 8. Model comparison on this dataset
**8.1** the chosen model of each type (chosen on val, scored on test). **8.2** every config tried, with its test score: it shows how much each model depends on its setting. Bold = chosen on val. A config that was not chosen can score higher on test; picking it would be tuning on the test set.

In [ ]:
cols = ["top1", "top3", "top5", "macro_precision", "macro_recall", "macro_f1", "mAP"]
summary = pd.DataFrame({m: r["test"] for m, r in RESULTS.items()}).T[cols]
summary.insert(0, "val_macro_f1", [RESULTS[m]["val"]["macro_f1"] for m in summary.index])
display(summary.style.highlight_max(axis=0, color="#cde4ff").format("{:.3f}"))
summary.to_csv(OUT / "summary_test.csv")

fig, ax = plt.subplots(figsize=(10, 4))
summary[["top1", "macro_f1", "mAP"]].plot.bar(ax=ax, rot=0)
ax.set_ylim(0, 1); ax.set_title(f"{DATASET}: test metrics by model"); ax.legend(loc="lower right")
plt.tight_layout(); savefig(fig, "model_comparison"); plt.show()

In [ ]:
# 8.2 every config tried (from all_models/, nothing is recomputed)
FAMILY = {"clip_zeroshot": "CLIP zero-shot", "logreg": "LogReg", "svm": "SVM", "cnn": "CNN", "resnet50": "ResNet-50"}
all_cfg = pd.DataFrame([{"family": FAMILY[r["family"]], "model": n, "chosen": CHOSEN.get(r["family"]) == n, "train_f1": r["train_f1"],
                         "val_macro_f1": r["val"]["macro_f1"], **{f"test_{k}": v for k, v in r["test"].items() if k != "n"}}
                        for n, r in CONFIGS.items()])
all_cfg.to_csv(OUT / "all_configs_test.csv", index=False)
num = [c for c in all_cfg.columns if c not in ("family", "model", "chosen")]
display(all_cfg.style.format({c: "{:.3f}" for c in num}, na_rep="–").background_gradient(cmap="Blues", subset=num[1:])
        .apply(lambda r: ["font-weight: bold" if r.chosen else ""] * len(r), axis=1))

fig, ax = plt.subplots(figsize=(8, 0.32 * len(all_cfg) + 1.2))
fam = {f: f"C{j}" for j, f in enumerate(all_cfg.family.unique())}
ax.barh(all_cfg.model, all_cfg.test_macro_f1, color=[fam[f] for f in all_cfg.family],
        edgecolor=["black" if c else "none" for c in all_cfg.chosen], linewidth=1.5)
for y, (v, c) in enumerate(zip(all_cfg.test_macro_f1, all_cfg.chosen)):
    ax.text(v + 0.005, y, f"{v:.3f}" + ("  (chosen on val)" if c else ""), va="center", fontsize=8)
ax.invert_yaxis(); ax.set_xlim(0, 1.15); ax.set_xlabel("test macro-F1")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in fam.values()], labels=list(fam), fontsize=8, loc="lower right")
ax.set_title(f"{DATASET}: test macro-F1 of every config tried (black edge = chosen on val)")
plt.tight_layout(); savefig(fig, "all_configs_test"); plt.show()

## 9. Grad-CAM: what each model looks at
Grad-CAM weights the last convolutional feature map by the gradient of the predicted class score, so red = regions that pushed the model toward its prediction.
- CNN: last conv block. ResNet-50: `layer4`. CLIP ViT: the 7×7 patch tokens of the last transformer block (the same idea applied to tokens).
- The classical models have no gradients, so they have no Grad-CAM.

Two image sets:
1. **Fixed set**: 1 test image per class, picked by seed from the manifest. It is identical in 01 / 02 / 03 / 04, so you can put the notebooks side by side.
2. Correct and wrong predictions of ResNet-50 (the strongest trained model).

In [ ]:
class GradCAM:
    def __init__(self, model, layer, tokens=False):
        self.model, self.tokens = model, tokens
        self.handle = layer.register_forward_hook(self._hook)

    def _hook(self, module, inp, out):
        self.acts = out
        out.register_hook(lambda g: setattr(self, "grads", g))

    def __call__(self, x):
        self.model.eval()
        x = x.to(DEVICE).requires_grad_(True)   # makes the hook fire even when the backbone is frozen
        logits = self.model(x)
        cls = logits.argmax(1)
        self.model.zero_grad(set_to_none=True)
        logits.gather(1, cls[:, None]).sum().backward()
        A, G = self.acts.detach(), self.grads.detach()
        if self.tokens:                         # (B, 1+N, C) ViT tokens -> (B, C, h, w), CLS token dropped
            side = int((A.shape[1] - 1) ** 0.5)
            A, G = [t[:, 1:].transpose(1, 2).reshape(t.shape[0], -1, side, side) for t in (A, G)]
        cam = F.relu((G.mean((2, 3), keepdim=True) * A).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
        cam = (cam - cam.amin((1, 2), keepdim=True)) / (cam.amax((1, 2), keepdim=True) - cam.amin((1, 2), keepdim=True) + 1e-8)
        return cam.cpu().numpy(), logits.detach().softmax(1).cpu().numpy()

    def remove(self):
        self.handle.remove()


cam_models = {   # name -> (model, target layer, transform, token input?)
    "CNN": (cnn_model, cnn_model.features[-2], eval_tf, False),
    "ResNet-50": (rn_model, rn_model.layer4, eval_tf, False),
    "CLIP": (clip_model, clip_model.clip.vision_model.encoder.layers[-1].layer_norm1, clip_tf, True),
}


def gradcam_grid(frame, title, name):
    cams = {m: GradCAM(model, layer, tok) for m, (model, layer, _, tok) in cam_models.items()}
    fig, axes = plt.subplots(len(frame), 1 + len(cams), figsize=(3 * (1 + len(cams)), 3.1 * len(frame)), squeeze=False)
    try:
        for row, (_, r) in zip(axes, frame.iterrows()):
            img = Image.open(r.path).convert("RGB")
            shown = np.asarray(pad_to_square(img).resize((IMG_SIZE, IMG_SIZE)))
            row[0].imshow(shown); row[0].set_title(f"true: {r.label}", fontsize=9)
            for ax, (m, cam) in zip(row[1:], cams.items()):
                heat, prob = cam(cam_models[m][2](img)[None])
                p = prob[0].argmax()
                ax.imshow(shown); ax.imshow(heat[0], cmap="jet", alpha=0.45)
                ax.set_title(f"{m}: {CLASSES[p]} ({prob[0, p]:.2f})", fontsize=9, color="green" if p == r.y else "red")
            for ax in row: ax.axis("off")
    finally:
        for c in cams.values(): c.remove()
    finish(fig, title, name)


fixed = splits["test"].groupby("label", group_keys=False).sample(1, random_state=SEED).reset_index(drop=True)
fixed[["relpath"]].to_csv(OUT / "gradcam_fixed_images.csv", index=False)
gradcam_grid(fixed, f"{DATASET}: Grad-CAM, fixed test images (same in 01-04)", "gradcam_fixed")

In [ ]:
rn_test = pd.read_csv(OUT / "resnet50" / "predictions_test.csv")
right_pool = splits["test"][rn_test.y_true.values == rn_test.y_pred.values]
right = right_pool.sample(min(5, len(right_pool)), random_state=SEED)
wrong_pool = splits["test"][rn_test.y_true.values != rn_test.y_pred.values]
wrong = wrong_pool.sample(min(5, len(wrong_pool)), random_state=SEED)
gradcam_grid(right.reset_index(drop=True), "Grad-CAM: ResNet-50 correct", "gradcam_correct")
if len(wrong):
    gradcam_grid(wrong.reset_index(drop=True), "Grad-CAM: ResNet-50 wrong", "gradcam_wrong")

## 10. Crop analysis (dataset 04 only)
Does the effect depend on what the DINO crop did to each image? (Skipped automatically on the other datasets.)
- **fallback vs cropped**: fallback = no food box, or boxes under 5% of the image, so the 02 image is used. There are only about 5 fallback test images (mostly miso soup), so that number is indicative only.
- accuracy by **box area** quintile (cropped images only): near 1.0 the crop kept almost the whole 02 image (about 30% of the images keep ≥ 90%); small values are tight crops, which may have cut food off.
- accuracy by **number of boxes**: all boxes are merged into one crop, so a crop built from several boxes usually still contains the table and other dishes.
- **10.2 mask (03) vs box (04) on the same test images.** The 03 mask check flagged about 16% of the images where the mask kept the plate and cut the food out.
  If the 03 results are in the same `results/` folder, this splits the test set by that flag and compares 03 and 04 accuracy in each group.
  If the box crop repairs the bad masks, 04 should gain most on "bad" and little on "ok". The example images also need the 03 images in `data/dataset/` (local run).

In [ ]:
if HAS_DINO:
    test_dn = splits["test"][["relpath"]].merge(dn, on="relpath", how="left")
    cropped = test_dn.fallback == 0
    test_dn["area_bin"] = pd.qcut(test_dn.box_area.where(cropped), 5, duplicates="drop")
    test_dn["boxes"] = pd.cut(test_dn.n_boxes.where(cropped), [0, 1, 2, np.inf], labels=["1", "2", "3+"])
    by_fb, by_area, by_boxes = {}, {}, {}
    for m in RESULTS:
        pr = pd.read_csv(OUT / m / "predictions_test.csv")
        correct = pd.Series((pr.y_true == pr.y_pred).values)
        by_fb[m] = correct.groupby(test_dn.fallback.values).mean()
        by_area[m] = correct.groupby(test_dn.area_bin.values, observed=True).mean()
        by_boxes[m] = correct.groupby(test_dn.boxes.values, observed=True).mean()
    fb_table = pd.DataFrame(by_fb).T.rename(columns={0: "cropped", 1: "fallback"})
    fb_table["n_fallback"] = int(test_dn.fallback.sum())
    display(fb_table)
    area_table = pd.DataFrame(by_area)
    area_table.index = [f"{iv.left:.2f}-{iv.right:.2f}" for iv in area_table.index]
    box_table = pd.DataFrame(by_boxes)
    box_n = test_dn.boxes.value_counts()
    display(area_table, box_table.assign(n=[int(box_n[i]) for i in box_table.index]))

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for m in RESULTS:
        axes[0].plot(range(len(area_table)), area_table[m], "o-", label=m)
        axes[1].plot(range(len(box_table)), box_table[m], "o-", label=m)
    axes[0].set_xticks(range(len(area_table)), area_table.index, rotation=20)
    axes[0].set_xlabel("box area quintile (fraction of the 02 image)"); axes[0].set_title("accuracy vs box area")
    axes[1].set_xticks(range(len(box_table)), [f"{i} (n={int(box_n[i])})" for i in box_table.index])
    axes[1].set_xlabel("food boxes merged into the crop"); axes[1].set_title("accuracy vs number of boxes")
    for ax in axes: ax.set_ylabel("test top-1"); ax.set_ylim(0, 1); ax.legend(fontsize=8)
    plt.tight_layout(); savefig(fig, "dino_crop_accuracy"); plt.show()

    small = test_dn[cropped].nsmallest(12, "box_area")
    pr = pd.read_csv(OUT / "resnet50" / "predictions_test.csv").set_index("relpath")
    fig, axes = plt.subplots(2, 6, figsize=(15, 6), squeeze=False)
    for ax in axes.ravel(): ax.axis("off")
    for ax, (_, r) in zip(axes.ravel(), small.iterrows()):
        p = pr.loc[r.relpath]
        ax.imshow(Image.open(DATA_ROOT / DATASET / r.relpath))
        ax.set_title(f"box {r.box_area:.2f}\nT: {p.label}\nP: {p.pred_label}", fontsize=8, color="green" if p.y_true == p.y_pred else "red")
    finish(fig, "tightest crops in the test set (ResNet-50 prediction)", "dino_tightest_crops")

In [ ]:
# 10.2 mask (03) vs box (04): the same test images, split by the 03 mask check
GSAM_DS = "03_denoise_clahe_gsam"
mc_path = OUT.parent / GSAM_DS / "mask_check.csv"
if HAS_DINO and not mc_path.exists():
    print(f"no 03 mask check in {mc_path.parent}: put the 03 results folder there to compare 03 and 04 image by image")
elif HAS_DINO:
    order = ["ok", "suspect", "bad", "fallback"]
    test_mc = splits["test"][["relpath"]].merge(pd.read_csv(mc_path)[["relpath", "mask_check"]], on="relpath", how="left")

    def right(path):
        p = pd.read_csv(path).set_index("relpath").loc[test_mc.relpath]
        return (p.y_true == p.y_pred).values

    rows = []
    for m in RESULTS:
        p03 = OUT.parent / GSAM_DS / m / "predictions_test.csv"
        if not p03.exists():
            continue
        r03, r04 = right(p03), right(OUT / m / "predictions_test.csv")
        for g in order:
            sel = (test_mc.mask_check == g).values
            if sel.any():
                rows.append({"model": m, "group": g, "n": int(sel.sum()), "acc_03": r03[sel].mean(), "acc_04": r04[sel].mean()})
    cmp = pd.DataFrame(rows)
    cmp["gain_04"] = cmp.acc_04 - cmp.acc_03
    cmp.to_csv(OUT / "mask_vs_box_by_mask_check.csv", index=False)
    groups = [g for g in order if g in set(cmp.group)]
    print("test images per 03 mask-check group:", test_mc.mask_check.value_counts().reindex(groups).to_dict())
    display(cmp.pivot(index="model", columns="group", values="gain_04")[groups].style.format("{:+.3f}")
            .background_gradient(cmap="RdBu", vmin=-0.4, vmax=0.4))

    models = list(dict.fromkeys(cmp.model))
    fig, axes = plt.subplots(1, len(models), figsize=(3.2 * len(models), 3.8), sharey=True, squeeze=False)
    for ax, m in zip(axes[0], models):
        t = cmp[cmp.model == m].set_index("group").loc[groups]
        x = np.arange(len(groups))
        ax.bar(x - 0.2, t.acc_03, 0.4, label="03 mask"); ax.bar(x + 0.2, t.acc_04, 0.4, label="04 box")
        ax.set_xticks(x, groups); ax.set_title(m); ax.set_ylim(0, 1)
    axes[0, 0].set_ylabel("test top-1"); axes[0, 0].legend(fontsize=8)
    finish(fig, "same test images: 03 (mask) vs 04 (box crop), by the 03 mask check", "mask_vs_box")

    if (DATA_ROOT / GSAM_DS).exists() and "resnet50" in models:
        preds = {ds: pd.read_csv(OUT.parent / ds / "resnet50" / "predictions_test.csv").set_index("relpath") for ds in (GSAM_DS, DATASET)}
        bad = test_mc[test_mc.mask_check == "bad"]
        bad = bad.sample(min(6, len(bad)), random_state=SEED)
        fig, axes = plt.subplots(2, max(1, len(bad)), figsize=(2.6 * max(1, len(bad)), 7.4), squeeze=False)
        for ax in axes.ravel(): ax.axis("off")
        for j, rel in enumerate(bad.relpath):
            for i, ds in enumerate(preds):
                r = preds[ds].loc[rel]
                axes[i, j].imshow(Image.open(DATA_ROOT / ds / rel))
                axes[i, j].set_title(f"{ds[:2]} | T: {r.label}\nP: {r.pred_label}", fontsize=8, color="green" if r.y_true == r.y_pred else "red")
        finish(fig, "test images with a bad 03 mask: 03 (top) vs 04 (bottom), ResNet-50", "mask_vs_box_examples")

## 11. Cross-dataset comparison: does segmentation or cropping help?
Runs on whichever of the 01 / 02 / 03 / 04 results are in `results/<dataset>/<model>/predictions_test.csv`
(run this notebook with each `DATASET`, or teammates copy their folders in using the same file format). Every dataset has the same test images, so the tests are **paired**:
- **McNemar exact test**: among the images only one of the two datasets classifies correctly, is the split uneven? p < 0.05 = a real difference in accuracy.
- **Bootstrap 95% CI** of the macro-F1 difference (1,000 resamples of the test images). If the CI excludes 0, the difference is reliable.

Pairs, B minus A: 01 → 02 (denoise + CLAHE), 02 → 03 (+ mask), **02 → 04 (+ box crop)**, **03 → 04 (mask replaced by box)**, and 01 → 03, 01 → 04 overall.

With about 1,920 test images, accuracy gaps under roughly 2 points are usually noise. 6 pairs × 5 models is 30 tests, so a single p ≈ 0.04 is weak evidence.

In [ ]:
from scipy.stats import binomtest

DATASETS = ["01_original", "02_denoise_clahe", "03_denoise_clahe_gsam", "04_denoise_clahe_dino"]
PAIRS = [("01_original", "02_denoise_clahe"), ("02_denoise_clahe", "03_denoise_clahe_gsam"), ("02_denoise_clahe", "04_denoise_clahe_dino"),
         ("03_denoise_clahe_gsam", "04_denoise_clahe_dino"), ("01_original", "03_denoise_clahe_gsam"), ("01_original", "04_denoise_clahe_dino")]
MODELS = ["clip_zeroshot", "logreg", "svm", "cnn", "resnet50"]


def load_preds(ds, model):
    p = OUT.parent / ds / model / "predictions_test.csv"
    return pd.read_csv(p).set_index("relpath").sort_index() if p.exists() else None


rows, per_ds = [], []
rng = np.random.default_rng(SEED)
for model in MODELS:
    preds = {ds: load_preds(ds, model) for ds in DATASETS}
    for ds, pr in preds.items():
        if pr is not None:
            per_ds.append({"model": model, "dataset": ds, "top1": (pr.y_true == pr.y_pred).mean(),
                           "macro_f1": f1_score(pr.y_true, pr.y_pred, average="macro")})
    for a, b in PAIRS:
        pa, pb = preds[a], preds[b]
        if pa is None or pb is None:
            continue
        assert pa.index.equals(pb.index), f"{a} and {b} have different test images"
        y, ya, yb = pa.y_true.values, pa.y_pred.values, pb.y_pred.values
        ca, cb = ya == y, yb == y
        n01, n10 = int((ca & ~cb).sum()), int((~ca & cb).sum())
        p_mc = binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
        boot = []
        for _ in range(1000):
            i = rng.integers(0, len(y), len(y))
            boot.append(f1_score(y[i], yb[i], average="macro") - f1_score(y[i], ya[i], average="macro"))
        rows.append({"model": model, "A": a, "B": b, "acc_A": ca.mean(), "acc_B": cb.mean(), "delta_acc": cb.mean() - ca.mean(),
                     "delta_macro_f1": f1_score(y, yb, average="macro") - f1_score(y, ya, average="macro"),
                     "f1_ci_low": np.percentile(boot, 2.5), "f1_ci_high": np.percentile(boot, 97.5),
                     "only_A_right": n01, "only_B_right": n10, "mcnemar_p": p_mc})

if per_ds:
    pivot = pd.DataFrame(per_ds).pivot(index="model", columns="dataset", values="macro_f1")
    display(pivot.style.format("{:.3f}").background_gradient(cmap="Blues", axis=1))
    fig, ax = plt.subplots(figsize=(10, 4))
    pivot.reindex(columns=[d for d in DATASETS if d in pivot.columns]).plot.bar(ax=ax, rot=0)
    ax.set_ylabel("test macro-F1"); ax.set_ylim(0, 1); ax.set_title("macro-F1 by model and dataset")
    plt.tight_layout(); savefig(fig, "cross_dataset"); plt.show()
if rows:
    paired = pd.DataFrame(rows)
    display(paired.style.format({c: "{:.3f}" for c in paired.columns if c not in ("model", "A", "B", "only_A_right", "only_B_right")}))
    paired.to_csv(OUT.parent / "paired_tests.csv", index=False)
else:
    print("only", sorted({r['dataset'] for r in per_ds}), "found so far; run the other datasets to compare")

## 12. Reload the saved models
Sections 4–7 already score the test set with the model each one just trained, still in memory. This section is separate: it rebuilds each final model **from its saved file**,
so it works in a fresh kernel, on another machine, or in a teammate's notebook. Run sections 0–3 first (settings, paths, imports, data, transforms, metric functions), then this section.
It does not need the training sections: the model classes and feature function are defined here again (identical to sections 5–7).

- `load_model(name, dataset)` reads `results/<dataset>/<name>/config.json` and rebuilds the model.
- **12.1** `predict_saved(name)` scores the whole test set with the reloaded model and checks the numbers against `metrics.json` from training. If they match, the saved file is correct.
- **12.2** `classify(path, name)` gives the top-k classes for one image.

Every model expects the same preprocessing as its dataset. A model trained on 04 expects a denoised, CLAHE'd, DINO-cropped image, so an outside photo
should go through the team's 02/04 pipeline first. The example uses a test image from the dataset, which is already preprocessed.

In [ ]:
import joblib
import cv2
from skimage.feature import hog, local_binary_pattern
from transformers import CLIPModel


class SimpleCNN(nn.Module):
    def __init__(self, n_blocks=4, base=32, dropout=0.3, n_classes=K):
        super().__init__()
        layers, c_in = [], 3
        for i in range(n_blocks):
            c = base * 2 ** i
            layers += [nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                       nn.Conv2d(c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                       nn.MaxPool2d(2)]
            c_in = c
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(dropout), nn.Linear(c_in, n_classes))

    def forward(self, x):
        return self.head(self.features(x))


def make_resnet50(unfreeze, pretrained=True):
    m = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
    m.fc = nn.Sequential(nn.Dropout(0.2), nn.Linear(m.fc.in_features, K))
    for p in m.parameters():
        p.requires_grad_(False)
    for part in {"fc": [m.fc], "layer4": [m.layer4, m.fc], "all": [m]}[unfreeze]:
        for p in part.parameters():
            p.requires_grad_(True)
    return m


class CLIPZeroShot(nn.Module):
    def __init__(self, clip, text_emb):
        super().__init__()
        self.clip = clip
        self.register_buffer("text_emb", text_emb)

    def forward(self, x):
        f = self.clip.visual_projection(self.clip.vision_model(pixel_values=x).pooler_output)
        return self.clip.logit_scale.exp() * F.normalize(f, dim=-1) @ self.text_emb.T


def handcrafted(path):
    rgb = np.asarray(pad_to_square(Image.open(path).convert("RGB")).resize((128, 128), Image.BILINEAR))
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    color = cv2.calcHist([hsv], [0, 1, 2], None, [8, 8, 8], [0, 180, 0, 256, 0, 256]).ravel()
    color /= color.sum() + 1e-8
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    shape = hog(gray, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2), block_norm="L2-Hys")
    texture = []
    for p, r in ((8, 1), (16, 2)):
        h, _ = np.histogram(local_binary_pattern(gray, p, r, method="uniform"), bins=p + 2, range=(0, p + 2))
        texture.append(h / h.sum())
    return np.concatenate([color, shape, *texture]).astype(np.float32)


_LOADED = {}   # (dataset, name) -> (model, config), so repeated calls don't reload from disk


def load_model(name, dataset=DATASET):
    """Rebuild a saved final model from results/<dataset>/<name>/. Returns (model, config)."""
    if (dataset, name) in _LOADED:
        return _LOADED[(dataset, name)]
    d = OUT.parent / dataset / name
    assert (d / "config.json").exists(), f"no saved model in {d}: run that model's section first"
    with open(d / "config.json") as f:
        cfg = json.load(f)
    assert cfg["classes"] == CLASSES, "saved model was trained on different classes"
    if cfg["kind"] == "sklearn":
        model = joblib.load(d / cfg["file"])
    else:
        ck = torch.load(d / cfg["file"], map_location="cpu")
        if cfg["kind"] == "cnn":
            model = SimpleCNN(**ck["arch"])
            model.load_state_dict(ck["state_dict"])
        elif cfg["kind"] == "resnet50":
            model = make_resnet50(ck["arch"]["unfreeze"], pretrained=False)
            model.load_state_dict(ck["state_dict"])
        elif cfg["kind"] == "clip":
            clip_m = CLIPModel.from_pretrained(ck["arch"]["clip_id"])
            model = CLIPZeroShot(clip_m, ck["text_emb"])
        model = model.to(DEVICE).eval()
    _LOADED[(dataset, name)] = (model, cfg)
    print(f"loaded {name} ({dataset}): val macro-F1 {cfg['val_macro_f1']}, test macro-F1 {cfg['test_macro_f1']}")
    return model, cfg


@torch.no_grad()
def resnet_embedding_one(img):
    if "embedder" not in _LOADED:
        m = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Identity()
        _LOADED["embedder"] = m.to(DEVICE).eval()
    return _LOADED["embedder"](eval_tf(img)[None].to(DEVICE)).cpu().numpy()


@torch.no_grad()
def predict_saved(name, split="test"):
    """Scores of a reloaded model on a whole split of this notebook's dataset -> (y, scores)."""
    model, cfg = load_model(name, DATASET)
    frame = splits[split]
    if cfg["kind"] == "sklearn":
        if cfg["arch"]["features"] == "handcrafted":
            from joblib import Parallel, delayed
            feats = np.stack(Parallel(n_jobs=-1, batch_size=32)(delayed(handcrafted)(p) for p in frame.path))
        else:
            feats = np.concatenate([resnet_embedding_one(Image.open(p).convert("RGB")) for p in frame.path])
        scores = model.predict_proba(feats) if hasattr(model, "predict_proba") else model.decision_function(feats)
    else:
        tf = clip_tf if cfg["kind"] == "clip" else eval_tf
        scores = np.concatenate([model(x.to(DEVICE)).float().softmax(1).cpu().numpy() for x, _ in make_loader(split, tf)])
    return frame.y.values, scores


@torch.no_grad()
def classify(path, name, dataset=DATASET, k=3, show=True):
    """Top-k prediction of one saved model for one image file."""
    model, cfg = load_model(name, dataset)
    img = Image.open(path).convert("RGB")
    if cfg["kind"] == "sklearn":
        feats = handcrafted(path)[None] if cfg["arch"]["features"] == "handcrafted" else resnet_embedding_one(img)
        scores = model.predict_proba(feats)[0] if hasattr(model, "predict_proba") else model.decision_function(feats)[0]
    else:
        tf = clip_tf if cfg["kind"] == "clip" else eval_tf
        scores = model(tf(img)[None].to(DEVICE)).float().softmax(1)[0].cpu().numpy()
    top = np.argsort(-scores)[:k]
    result = [(CLASSES[i], float(scores[i])) for i in top]
    if show:
        fig, ax = plt.subplots(figsize=(3.2, 3.6))
        ax.imshow(img); ax.axis("off")
        ax.set_title(f"{name} ({dataset[:2]})\n" + "\n".join(f"{c}: {v:.2f}" for c, v in result), fontsize=9)
        plt.show()
    return result

In [ ]:
# 12.1 whole test set from the saved files, compared with the numbers recorded at training time
rows = []
for name in ["clip_zeroshot", "logreg", "svm", "cnn", "resnet50"]:
    if not (OUT / name / "config.json").exists():
        print(f"{name}: not saved yet"); continue
    y, sc = predict_saved(name, "test")
    m, _ = compute_metrics(y, sc)
    with open(OUT / name / "metrics.json") as f:
        trained = json.load(f)["test"]
    rows.append({"model": name, **{k: m[k] for k in ("top1", "top3", "top5", "macro_f1", "mAP")},
                 "macro_f1_at_training": trained["macro_f1"],
                 "match": abs(m["macro_f1"] - trained["macro_f1"]) < 1e-3})
if rows:
    reloaded = pd.DataFrame(rows).set_index("model")
    display(reloaded)
    if not reloaded.match.all():
        print("a reloaded model differs from its training-time result: check that the dataset and settings are the same as when it was saved")

In [ ]:
# 12.2 one image: every saved model on the same test image (swap in any image path)
example = splits["test"].sample(1, random_state=SEED).iloc[0]
print("true label:", example.label, "|", example.relpath)
for name in ["clip_zeroshot", "logreg", "svm", "cnn", "resnet50"]:
    if (OUT / name / "config.json").exists():
        print(f"{name:14s}", classify(example.path, name, show=False))
    else:
        print(f"{name:14s} not saved yet")
if (OUT / "resnet50" / "config.json").exists():
    classify(example.path, "resnet50")

## 13. Checklist for the report

- [ ] Data: same manifest hash as the team, balanced counts, cleaning summary (`removed.csv`), cross-split duplicate check, what the DINO crop did (box area, boxes, fallbacks)
- [ ] Per model: chosen hyper-parameters + the train/val graph that justifies them; whether training early-stopped or hit the epoch limit; final model saved
- [ ] Test table: top-1/3/5, macro P/R/F1, mAP; per-class table; confusion matrix + most-confused pairs
- [ ] Every config tried (8.2): how much each model depends on its setting (test shown, never used to choose)
- [ ] Correct / wrong examples and Grad-CAM (fixed images compared across 01–04)
- [ ] Crop: fallback vs cropped, accuracy vs box area and number of boxes, 03 vs 04 on the 03 mask-check groups (10.2)
- [ ] Cross-dataset: macro-F1 per model × dataset, paired McNemar + bootstrap CI, especially 02 → 04 (does the crop help?) and 03 → 04 (mask or box?)
- [ ] Limits: 1 seed (re-run CNN/ResNet with 2–3 more seeds if time allows), label noise in Food-101, about 5 fallback test images only, CNN may hit its 40-epoch limit (same limit on every dataset)